# Continuous CPS — scikit-learn and LightGBM

Cross-fitted, locally scaled CPS with CDF, PPF, quantiles, coverage, and Newsvendor/capacity decisions. For LightGBM: `pip install lightgbm`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyconformal.distribution import ContinuousCrossConformalPredictiveSystem
from tinyconformal.utils import NewsvendorSolver

rng = np.random.default_rng(42)
X = rng.uniform(0, 10, size=(3000, 1))
y = 20 + 3 * X[:, 0] + rng.normal(0, 1 + 0.4 * X[:, 0])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [2]:
scale_model = RandomForestRegressor(n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1)
model= RandomForestRegressor(n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1)
cps = ContinuousCrossConformalPredictiveSystem(model, scale_model, cv=5, n_jobs=-1).fit(X_train, y_train)
distribution = cps.predict_distribution(X_test)
display(distribution.evaluate(y_test))

,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.468333,3.921675,7.768870
1,0.80,0.776667,8.108083,10.781013
2,0.90,0.895000,10.758851,12.634166
3,0.95,0.965000,13.291093,14.711845


## CDF, PPF, and quantile predictions

In [3]:
quantile_levels = np.array([0.1, 0.5, 0.9])
quantile_predictions = distribution.ppf(quantile_levels)

summary = pd.DataFrame({
    "y": y_test[:10],
    "cdf_at_y": distribution.cdf(y_test[:, None])[:10],
    "q10": quantile_predictions[:10, 0],
    "q50": quantile_predictions[:10, 1],
    "q90": quantile_predictions[:10, 2],
})
summary

,y,cdf_at_y,q10,q50,q90
0,37.349480,0.289046,34.684930,39.194805,43.710518
1,34.458868,0.468555,31.879232,34.598010,37.320308
2,36.680762,0.898376,31.159510,33.932728,36.709537
3,49.109402,0.790087,40.815397,46.105729,51.402910
4,21.991888,0.652228,20.269961,21.624485,22.980763
5,47.424725,0.852978,34.264498,41.664922,49.074926
6,46.529311,0.264473,43.821467,48.844902,53.874842
7,41.513596,0.824656,30.182845,36.942849,43.711605
8,24.281967,0.734277,22.157144,23.638158,25.121089
9,40.535135,0.052895,41.901780,46.378071,50.860158


## Solver

For continuous outcomes, the solver can represent optimal capacity. The row order must match the distribution.

In [4]:
decision_frame = pd.DataFrame({
    "unique_id": np.arange(len(y_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "excess_cost": 1.0,
})
solver_result = NewsvendorSolver.optimize_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="excess_cost",
)
solver_result.head()

,unique_id,ds,shortage_cost,excess_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,43.710518
1,1,2026-01-01,9.0,1.0,0.9,37.320308
2,2,2026-01-01,9.0,1.0,0.9,36.709537
3,3,2026-01-01,9.0,1.0,0.9,51.402910
4,4,2026-01-01,9.0,1.0,0.9,22.980763
